# 01 — Exploration des données · Databricks
**Projet** : FakeNews Analyzer
**Exécuter après** : upload CSV sur `dbfs:/FileStore/fakenews/raw/`
---

In [ ]:
%pip install wordcloud vaderSentiment seaborn -q

In [ ]:
# ============================================================
# Setup Databricks — import spark_utils par chemin absolu
# Compatible /Users/ et /Repos/ (Databricks Workspace moderne)
# ============================================================
import sys, os, importlib.util

# 1. Résolution du repo root
#    Le notebook est dans .../mon-repo/databricks/nom_notebook
#    On remonte d'un niveau par rapport au dossier 'databricks'
_ctx   = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
_parts = _ctx.split('/')

# Chercher le dossier 'databricks' dans le path et prendre tout ce qui est avant
_db_idx = next((i for i, p in enumerate(_parts) if p == 'databricks'), None)
if _db_idx is not None:
    _REPO = '/Workspace/' + '/'.join(_parts[1:_db_idx])
else:
    # Fallback : remonter 2 niveaux depuis le notebook
    _REPO = '/Workspace/' + '/'.join(_parts[1:-2])

_UTILS_FILE = _REPO + '/02_preprocessing/spark_utils.py'

print(f'Notebook path : {_ctx}')
print(f'Repo root     : {_REPO}')
print(f'spark_utils   : {_UTILS_FILE}')

if not os.path.exists(_UTILS_FILE):
    raise FileNotFoundError(
        f"spark_utils.py introuvable : {_UTILS_FILE}
"
        f"Verifier que le repo contient bien 02_preprocessing/spark_utils.py"
    )

# 2. Import direct par chemin absolu (bypass du package pip 'spark_utils')
_spec = importlib.util.spec_from_file_location('_spark_utils', _UTILS_FILE)
_mod  = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)

load_raw_sources        = _mod.load_raw_sources
show_label_distribution = _mod.show_label_distribution
save_parquet            = _mod.save_parquet
load_parquet            = _mod.load_parquet
stratified_split        = _mod.stratified_split
check_class_balance     = _mod.check_class_balance

# 3. Chemins Volume Unity Catalog
# Trouver le chemin : Databricks > Data > Volumes > clic droit "fakenews" > Copy path
VOLUME        = '/Volumes/main/default/fakenews'  # <- MODIFIER si catalog/schema different

RAW_DIR       = VOLUME + '/raw'
PROCESSED_DIR = VOLUME + '/processed'
MODELS_DIR    = VOLUME + '/models'
COLAB_DIR     = VOLUME + '/colab_exports'
REPORTS_DIR   = VOLUME + '/reports'

for d in [PROCESSED_DIR, MODELS_DIR, COLAB_DIR, REPORTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Volume        : {VOLUME}')
print('Setup OK')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
plt.style.use('dark_background')
print('Imports OK')

## Section 1 — Chargement des sources

In [ ]:
# spark est fourni par le cluster — pas besoin de get_spark_session()
sources = load_raw_sources(spark, RAW_DIR)
print(f'Sources chargees : {list(sources.keys())}')

## Section 2 — Stats de base

In [ ]:
for name, df in sources.items():
    print(f'\n{"="*50}')
    print(f'Source : {name}  |  Lignes : {df.count():,}  |  Colonnes : {len(df.columns)}')
    df.printSchema()
    display(df.limit(3))

## Section 3 — Distribution des labels

In [ ]:
from pyspark.sql import functions as F

for name, df in sources.items():
    label_candidates = [c for c in df.columns if c.lower() in ('label', 'fraudulent', 'fake')]
    if not label_candidates:
        print(f'[{name}] Pas de colonne label detectee')
        continue
    label_col = label_candidates[0]
    print(f'\n[{name}] — colonne label : "{label_col}"')
    show_label_distribution(df, label_col)

In [ ]:
fig, axes = plt.subplots(1, max(len(sources), 1), figsize=(5 * len(sources), 5))
if len(sources) == 1:
    axes = [axes]
for ax, (name, df) in zip(axes, sources.items()):
    label_candidates = [c for c in df.columns if c.lower() in ('label', 'fraudulent')]
    if not label_candidates:
        continue
    label_col = label_candidates[0]
    counts = df.groupBy(label_col).count().toPandas()
    counts['label_name'] = counts[label_col].apply(lambda x: 'FAKE' if str(x) in ('1','FAKE','fake') else 'REAL')
    colors = ['#ff4444' if l == 'FAKE' else '#00ccaa' for l in counts['label_name']]
    ax.bar(counts['label_name'], counts['count'], color=colors, edgecolor='white', linewidth=0.5)
    ax.set_title(f'Distribution — {name}', fontsize=12)
    ax.set_ylabel("Nombre d'articles")
    for bar, cnt in zip(ax.patches, counts['count']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f'{cnt:,}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, '01_label_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

## Section 4 — Longueurs de texte

In [ ]:
for name, df in sources.items():
    text_col = next((c for c in df.columns if c.lower() in ('text','statement','tweet','content')), None)
    if not text_col:
        continue
    df_len = df.withColumn('text_len', F.length(F.col(text_col))).select('text_len')
    stats = df_len.describe().toPandas()
    print(f'\n[{name}] Longueurs de texte (caracteres) :')
    display(stats)

## Section 5 — Top 20 mots FAKE vs REAL

In [ ]:
# Top 20 mots — Python pur (RegexTokenizer/StopWordsRemover bloqués en cluster Shared)
import re
from collections import Counter

STOPWORDS_SET = {
    'the','a','an','and','or','but','in','on','at','to','for','of','with',
    'is','are','was','were','be','been','being','have','has','had','do','does',
    'did','will','would','could','should','may','might','shall','can','need',
    'i','me','my','we','our','you','your','he','him','his','she','her',
    'they','them','their','this','that','these','those','what','which','who',
    'not','no','nor','so','than','too','very','just','from','up','about','into'
}

def top_words_python(df, text_col, label_col, label_value, n=20):
    subset = df.filter(F.col(label_col).cast('string') == str(label_value))
    sample_size = min(5000, subset.count())
    texts = subset.select(text_col).limit(sample_size).toPandas()[text_col].tolist()
    counter = Counter()
    for text in texts:
        if not isinstance(text, str):
            continue
        tokens = re.findall(r'[a-z]{3,}', text.lower())
        counter.update(t for t in tokens if t not in STOPWORDS_SET)
    top = counter.most_common(n)
    return pd.DataFrame(top, columns=['word', 'count'])

for name, df in sources.items():
    text_col  = next((c for c in df.columns if c.lower() in ('text','statement')), None)
    label_col = next((c for c in df.columns if c.lower() in ('label','fraudulent')), None)
    if not text_col or not label_col:
        continue
    print(f'Calcul des top mots pour {name}...')
    fake_words = top_words_python(df, text_col, label_col, 1)
    real_words = top_words_python(df, text_col, label_col, 0)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    ax1.barh(fake_words['word'][::-1], fake_words['count'][::-1], color='#ff4444')
    ax1.set_title('Top 20 mots — FAKE')
    ax2.barh(real_words['word'][::-1], real_words['count'][::-1], color='#00ccaa')
    ax2.set_title('Top 20 mots — REAL')
    plt.tight_layout()
    plt.show()
    break

## Section 6 — Rapport de qualité

In [ ]:
MIN_TEXT_LENGTH = 20
quality_report = []
for name, df in sources.items():
    text_col = next((c for c in df.columns if c.lower() in ('text','statement','tweet')), None)
    if not text_col:
        continue
    total     = df.count()
    nulls     = df.filter(F.col(text_col).isNull()).count()
    empty     = df.filter(F.trim(F.col(text_col)) == '').count()
    too_short = df.filter(F.length(F.col(text_col)) < MIN_TEXT_LENGTH).count()
    quality_report.append({
        'source': name, 'total': total, 'nulls': nulls, 'empty': empty,
        'too_short': too_short, 'exploitable': total - nulls - empty,
        'qualite_%': round((total - nulls - empty) / total * 100, 1) if total > 0 else 0
    })
report_df = pd.DataFrame(quality_report)
display(report_df)
report_df.to_csv(os.path.join(REPORTS_DIR, '01_quality_report.csv'), index=False)
print(f'Rapport sauvegarde : {REPORTS_DIR}/01_quality_report.csv')
print('Prochaine etape : 02_cleaning_db.ipynb')